# 真实 Qwen：一句话的 token 概率
在读完主课后运行。本文件使用已缓存的 Qwen3-0.6B，没有下载、训练或云服务。
选择项目 .venv 内核，需要现有 inference 依赖。只做一次短回答，最长 8 个 token；输出可能随硬件变化，不人为替换模型结果。

为便于对应公式，手写一个很短的生成循环；不启用温度缩放或 top-k/top-p 截断。每一步直接按照 softmax 概率采样，计算出的就是本次实际采样分布的概率。

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import snapshot_download

path = snapshot_download("Qwen/Qwen3-0.6B", local_files_only=True)
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32 if device == "cpu" else torch.float16
tokenizer = AutoTokenizer.from_pretrained(path, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(path, dtype=dtype, local_files_only=True).to(device).eval()
print("设备：", device)

设备： mps


In [2]:
messages = [{"role": "user", "content": "2+2等于多少？只回答一个数字，不要解释。"}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt",
                                      add_generation_prompt=True, enable_thinking=False).to(device)
print("输入 token 数：", inputs.shape[1])
print(tokenizer.decode(inputs[0]))

输入 token 数： 26
<|im_start|>user
2+2等于多少？只回答一个数字，不要解释。<|im_end|>
<|im_start|>assistant
<think>

</think>




每步都让完整前文过一次模型，取最后一个位置的 logits。为了容易看懂，这里不用 KV cache，短回答可接受，但不是高性能推理写法。

In [3]:
torch.manual_seed(123)
prefix = inputs
generated = []
log_probs = []
eos_ids = model.generation_config.eos_token_id
eos_ids = {eos_ids} if isinstance(eos_ids, int) else set(eos_ids or [tokenizer.eos_token_id])

with torch.no_grad():  # 本文件只观察，不构建训练计算图
    for step in range(8):
        logits = model(input_ids=prefix, use_cache=False).logits[0, -1].float()
        dist = torch.distributions.Categorical(logits=logits)
        token = dist.sample()
        log_p = dist.log_prob(token).item()
        token_id = token.item()
        generated.append(token_id)
        log_probs.append(log_p)
        print(step + 1, "id =", token_id, "文字 =", repr(tokenizer.decode([token_id])),
              f"p = {torch.exp(torch.tensor(log_p)).item():.6f}", f"log p = {log_p:.6f}")
        prefix = torch.cat([prefix, token.reshape(1, 1)], dim=1)
        if token_id in eos_ids:
            break

finished = generated[-1] in eos_ids
answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
score = float(finished and answer == "4")
print("回答：", repr(answer))
print("正常结束：", finished, "（False 表示只观察到一个截断前缀）")
print("奖励：", score)
print("生成部分 log 概率之和（包含实际采样的结束符）：", sum(log_probs))

1 id = 17 文字 = '2' p = 0.978269 log p = -0.021971
2 id = 151645 文字 = '<|im_end|>' p = 0.994750 log p = -0.005264
回答： '2'
正常结束： True （False 表示只观察到一个截断前缀）
奖励： 0.0
生成部分 log 概率之和（包含实际采样的结束符）： -0.027235031127929688


这些概率来自真实模型的前向计算。没有正常结束时，上面的和只是已生成前缀的 log 概率，不能叫完整回答概率。

如果训练，就要保留或重新计算可求导的 log 概率，构造奖励加权 loss，再 backward / step。本格用了 no_grad 和 item，已经切断梯度；不能直接把这里打印出来的数字拿去训练。

只记录模型生成部分；用户问题、模板等条件输入的概率不加入这个策略梯度 loss。不同样本有不同停止位置时，各自只计到自己的结束符。

本机本次实测，模型输出的是 **2**，随后生成结束符，奖励为 **0**。第一个 token 的概率约 0.9783，但答案仍然错了。高概率表示模型倾向于这样生成，不是外部判定的正确率。保留这个错误输出，用它区分模型概率与奖励；换硬件或采样结果变化时，以你实际输出为准。

In [4]:
# 学完释放这份 Notebook 持有的模型内存
import gc
del model
gc.collect()
if device == "mps":
    torch.mps.empty_cache()
elif device == "cuda":
    torch.cuda.empty_cache()
print("已释放本 Notebook 的模型引用。")

已释放本 Notebook 的模型引用。
